In [14]:
import tensorflow as tf
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import numpy as np
import keras
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.utils import register_keras_serializable
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from glob import glob
import rasterio
import pandas as pd
import geopandas as gpd
from rasterio.transform import from_origin


## Functions for estimation

In [ ]:
# Make sure dropout layers are active
def mc_dropout_predict(
    model,
    X,
    n_mc=50,
    batch_size=1024
):
    preds = []

    for _ in range(n_mc):
        batch_preds = []

        for Xb in batch_generator(X, batch_size):
            yb = model(Xb, training=True).numpy()
            batch_preds.append(yb)

        batch_preds = np.concatenate(batch_preds, axis=0)
        preds.append(batch_preds)

    preds = np.stack(preds, axis=0)  # (n_mc, N, 1)

    mean = preds.mean(axis=0).squeeze()
    std = preds.std(axis=0).squeeze()

    return mean, std

def batch_generator(X, batch_size=1024):
    N = X.shape[0]
    for i in range(0, N, batch_size):
        yield X[i:i+batch_size]

def get_property_name(model_path):
    name = os.path.basename(model_path).lower()

    for key in ["sand", "silt", "clay"]:
        if key in name:
            return key

    return os.path.splitext(name)[0]

def save_raster(path, array, meta, dtype):
    meta_out = meta.copy()
    meta_out.update(dtype=dtype, count=1)

    with rasterio.open(path, "w", **meta_out) as dst:
        dst.write(array.astype(dtype), 1)


In [13]:
def channel_attention(input_feature, ratio=8, name=None):
    channel = input_feature.shape[-1]
    # Shared MLP layers
    shared_dense_one = layers.Dense(channel // ratio,
                                    activation='relu',
                                    kernel_initializer='he_normal',
                                    use_bias=True,
                                    bias_initializer='zeros',
                                    name=f'{name}_mlp_0' if name else None)
    shared_dense_two = layers.Dense(channel,
                                    kernel_initializer='he_normal',
                                    use_bias=True,
                                    bias_initializer='zeros',
                                    name=f'{name}_mlp_1' if name else None)
    # Global average pooling
    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_two(shared_dense_one(avg_pool))
    # Global max pooling
    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_two(shared_dense_one(max_pool))
    # Combine and scale
    scale = layers.Activation('sigmoid')(avg_pool + max_pool)
    return layers.Multiply()([input_feature, scale])


In [14]:
@register_keras_serializable()
class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, name=None, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)
        self.kernel_size = kernel_size
        # Give the conv layer a name if provided
        self.conv = layers.Conv2D(
            1, 
            kernel_size=self.kernel_size, 
            strides=1,
            padding='same', 
            activation='sigmoid', 
            use_bias=False,
            name=f"{name}_conv" if name else None
        )

    def call(self, inputs):
        # mean and max across channels
        avg_pool = K.mean(inputs, axis=-1, keepdims=True)
        max_pool = K.max(inputs, axis=-1, keepdims=True)
        concat = layers.Concatenate(axis=-1)([avg_pool, max_pool])
        cbam_feature = self.conv(concat)
        return inputs * cbam_feature

# Prepare the data 

In [16]:
scaler = StandardScaler()
inputs = np.load("../data/inputs.npy")
X_full_scaled = scaler.fit_transform(inputs.reshape(-1, inputs.shape[-1])).reshape(inputs.shape)

In [18]:
X_full_scaled.shape

(2156, 7, 7, 38)

## Main 

In [ ]:
# Paths
MODEL_DIR = "models_utcnn"
OUT_DIR = "outputs_uncertainty_utcnn"
os.makedirs(OUT_DIR, exist_ok=True)

# Load reference raster metadata (for georeferencing)
with rasterio.open("cnn_basic_maps\A\A_map.tif") as ref:
    meta = ref.meta.copy()
    rows, cols = ref.height, ref.width

# X must match rows*cols
# assert X_full_scaled.shape[0] == rows * cols, "X does not match raster grid"

# Loop through models
for model_path in glob(os.path.join(MODEL_DIR, "*.keras")):
    prop = get_property_name(model_path)
    print(f"\nProcessing property: {prop}")

    model = keras.models.load_model(model_path, compile=False)

    mean, std = mc_dropout_predict(
        model,
        X_full_scaled,
        n_mc=100,
        batch_size=1024
    )

    df_prop=pd.DataFrame([])
    df_prop['mean_unc'] = mean
    df_prop['std_unc'] = std
    df_prop.to_csv(f'unceratainty_{prop}.csv')
    
    # Reshape to raster
    # mean_r = mean.reshape(rows, cols)
    # std_r = std.reshape(rows, cols)

    # Save
    # save_raster(
    #     os.path.join(OUT_DIR, f"{prop}_mean.tif"),
    #     mean_r,
    #     meta,
    #     rasterio.float32
    # )

    # save_raster(
    #     os.path.join(OUT_DIR, f"{prop}_std.tif"),
    #     std_r,
    #     meta,
    #     rasterio.float32
    # )

    print(f"Saved {prop}_mean.tif and {prop}_std.tif")


In [ ]:
# Load your trained model
import keras

model = keras.models.load_model("models_utcnn/final_model_outputA%.keras")
model.summary()


c:\Users\ASUS\anaconda3\envs\dsm_env\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'spatial_attention', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_11      │ (None, 7, 7, 38)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_38 (Conv2D)  │ (None, 7, 7, 128) │     43,904 │ input_layer_11[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_38[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv2d_38[0][0]   │
│ (GlobalMaxPooling2… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_36          │ (None, 1, 1, 128) │          0 │ global_average_p… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_37          │ (None, 1, 1, 128) │          0 │ global_max_pooli… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cbam_block_1_ch_ml… │ (None, 1, 1, 10)  │      1,290 │ reshape_36[0][0], │
│ (Dense)             │                   │            │ reshape_37[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cbam_block_1_ch_ml… │ (None, 1, 1, 128) │      1,408 │ cbam_block_1_ch_… │
│ (Dense)             │                   │            │ cbam_block_1_ch_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_18 (Add)        │ (None, 1, 1, 128) │          0 │ cbam_block_1_ch_… │
│                     │                   │            │ cbam_block_1_ch_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_18       │ (None, 1, 1, 128) │          0 │ add_18[0][0]      │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_18         │ (None, 7, 7, 128) │          0 │ conv2d_38[0][0],  │
│ (Multiply)          │                   │            │ activation_18[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_attention   │ (None, 7, 7, 128) │         98 │ multiply_18[0][0] │
│ (SpatialAttention)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_13    │ (None, 3, 3, 128) │          0 │ spatial_attentio… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_46          │ (None, 3, 3, 128) │          0 │ max_pooling2d_13… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_40 (Conv2D)  │ (None, 3, 3, 128) │    802,944 │ dropout_46[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_14    │ (None, 1, 1, 128) │          0 │ conv2d_40[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_47          │ (None, 1, 1, 128) │          0 │ max_pooling2d_14… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 2,576,087 (9.83 MB)

 Trainable params: 858,695 (3.28 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,717,392 (6.55 MB)

In [19]:
y1 = model(X_full_scaled, training=True).numpy()
y2 = model(X_full_scaled, training=True).numpy()

np.allclose(y1, y2)

False

In [ ]:
import rasterio
import numpy as np
import plotly.graph_objects as go


def plot_uncertainty_map(
    raster_path,
    title="Uncertainty Map",
    colorscale="Viridis",
    output_html=None,
    width=900,
    height=700
):
    """
    Visualize a continuous uncertainty (std) raster using Plotly.

    Parameters
    ----------
    raster_path : str
        Path to uncertainty GeoTIFF (e.g., sand_std.tif)
    title : str
        Plot title
    colorscale : str or list
        Plotly colorscale (default: Viridis)
    output_html : str or None
        If provided, saves interactive HTML to this path
    width : int
        Figure width in pixels
    height : int
        Figure height in pixels
    """

    # Load raster
    with rasterio.open(raster_path) as src:
        data = src.read(1).astype(float)
        bounds = src.bounds
        nodata = src.nodata

    # Mask NoData
    if nodata is not None:
        data[data == nodata] = np.nan

    # Build geographic coordinates
    x = np.linspace(bounds.left, bounds.right, data.shape[1])
    y = np.linspace(bounds.top, bounds.bottom, data.shape[0])

    # Create plot
    fig = go.Figure(go.Heatmap(
        z=data,
        x=x,
        y=y,
        colorscale=colorscale,
        colorbar=dict(
            title="Uncertainty (Std)",
            titleside="right"
        ),
        hovertemplate=(
            "Uncertainty (std): %{z:.3f}<br>"
            "Lon: %{x:.4f}<br>"
            "Lat: %{y:.4f}<br>"
            "<extra></extra>"
        )
    ))

    fig.update_layout(
        title=title,
        yaxis_autorange="reversed",
        width=width,
        height=height
    )

    # Save to HTML if requested
    if output_html is not None:
        fig.write_html(output_html, include_plotlyjs=True)
        print(f"Saved interactive map to: {output_html}")

    fig.show()

plot_uncertainty_map()